# 51 — Explainable Scoring
**Goal:** Generate human-readable explanations for each resume score.

## 1. Score Breakdown with Explanations

In [ ]:
class ExplainableScorer(ATSScorer):
    def score_with_explanations(self, resume_text, jd_text, resume_skills, jd_skills,
                                  resume_years=5, required_years=3, must_have_terms=None):
        explanations = {}
        
        # Skill match
        s = self.skill_match_score(resume_skills, jd_skills)
        explanations["skill_match"] = {
            "score": round(s, 1),
            "reason": f"Matched {int(s/100*len(jd_skills)) if jd_skills else 0}/{len(jd_skills)} required skills",
            "details": {"found": resume_skills, "required": jd_skills}
        }
        
        s2 = self.experience_score(resume_years, required_years)
        explanations["experience"] = {
            "score": round(s2, 1),
            "reason": f"{resume_years} years experience vs {required_years} required"
        }
        
        s3 = self.format_score(resume_text)
        missing = []
        for sec in ["summary", "experience", "education", "skills"]:
            if not re.search(r"\\b" + sec + r"\\b", resume_text, re.IGNORECASE):
                missing.append(sec)
        explanations["format"] = {
            "score": round(s3, 1),
            "reason": f"Missing sections: {missing}" if missing else "All sections present"
        }
        
        if must_have_terms:
            s4 = self.boolean_check(resume_text, must_have_terms)
            explanations["boolean"] = {
                "score": round(s4, 1),
                "reason": f"Found {int(s4/100*len(must_have_terms))}/{len(must_have_terms)} required terms"
            }
        
        total = sum(ex["score"] * self.weights[k] for k, ex in explanations.items())
        return round(total, 1), explanations

es = ExplainableScorer()
score, exps = es.score_with_explanations(
    "Data scientist with Python, NLP, TensorFlow. 5 years. MS Computer Science.",
    "Senior data scientist, Python NLP required, 5+ years",
    ["Python", "NLP", "TensorFlow"], ["Python", "NLP", "SQL"],
    must_have_terms=["Python", "ML"]
)
print(f"Total score: {score}/100\n")
for dim, exp in exps.items():
    print(f"  [{dim:15s}] {exp['score']:5.1f}/100 | {exp['reason']}")

## 2. Visualization of Score Breakdown

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

dims = list(exps.keys())
scores = [exps[d]['score'] for d in dims]
weights = [es.weights.get(d, 0.1) for d in dims]
weighted = [s * w for s, w in zip(scores, weights)]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.bar(dims, scores, color='steelblue')
ax1.set_ylabel('Raw Score (0-100)')
ax1.set_title('Dimension Scores')
ax1.tick_params(axis='x', rotation=45)

ax2.bar(dims, weighted, color='coral')
ax2.set_ylabel('Weighted Contribution')
ax2.set_title('Final Contribution (score x weight)')
ax2.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig('/tmp/ats_breakdown.png', dpi=100)
plt.show()
print("\nChart saved to /tmp/ats_breakdown.png")

## Summary: Every score has an explanation. Transparency builds trust with recruiters and candidates.